# 🔧 Clase 5 — Features, encoding y partición de datos

Tenemos datos limpios. Ahora necesitamos **transformarlos** para que un modelo de ML pueda usarlos.
En esta clase aprendés a crear features, codificar categorías y dividir datos en train/validation/test.

### ¿Qué vamos a cubrir?

| Sección | Concepto |
|---|---|
| 1 | Setup y regenerar dataset limpio |
| 2 | ¿Qué son las features? |
| 3 | Features numéricas vs categóricas |
| 4 | One-hot encoding |
| 5 | Normalización |
| 6 | Train / Validation / Test |
| 7 | Primera predicción con regresión lineal |
| 8 | Ejercicio evaluable |

> **Los modelos de ML ven números, no palabras.** Tenemos que convertir todo a números.

In [ ]:
# ── 1. Setup ────────────────────────────────────────────────

import pandas as pd
import matplotlib.pyplot as plt
import random

random.seed(42)

print(f"✅ pandas     : {pd.__version__}")
print(f"✅ matplotlib : {plt.matplotlib.__version__}")

In [ ]:
# Regeneramos el dataset limpio (mismo que clase 3, sin errores)

random.seed(42)

nombres_pool = [
    "Ana", "Lucía", "Martín", "Sofía", "Mateo", "Valentina", "Santiago",
    "Isabella", "Sebastián", "Camila", "Nicolás", "Emilia", "Daniel",
    "Renata", "Tomás", "Paula", "Alejandro", "Florencia", "Lucas",
]
apellidos_pool = [
    "García", "Martínez", "López", "González", "Rodríguez", "Pérez",
    "Sánchez", "Ramírez", "Torres", "Flores",
]
carreras = ["Ingeniería", "Sistemas", "Diseño", "Economía"]
N = 200
datos = []

for i in range(N):
    nombre = f"{random.choice(nombres_pool)} {random.choice(apellidos_pool)}"
    edad = random.randint(18, 35)
    carrera = random.choice(carreras)
    trabaja = random.random() < 0.4
    base_horas = random.gauss(12, 5) if not trabaja else random.gauss(7, 4)
    horas_estudio = round(max(0, min(30, base_horas)), 1)
    asistencia = round(min(100, max(0, horas_estudio * 4.5 + random.gauss(20, 15))), 1)
    nota_parcial = round(min(10, max(0,
        horas_estudio * 0.3 + asistencia * 0.03 + random.gauss(1, 1.5))), 1)
    nota_final = round(min(10, max(0,
        nota_parcial * 0.4 + horas_estudio * 0.2 + asistencia * 0.02 + random.gauss(0.5, 1))), 1)
    datos.append({
        "nombre": nombre, "edad": edad, "carrera": carrera,
        "horas_estudio": horas_estudio, "nota_parcial": nota_parcial,
        "asistencia_pct": asistencia, "trabaja": trabaja, "nota_final": nota_final,
    })

df = pd.DataFrame(datos)
print(f"✅ Dataset limpio: {len(df)} filas × {len(df.columns)} columnas")
df.head(3)

---
## 2. ¿Qué son las features?

En Machine Learning:

| Término | Significado | En nuestro dataset |
|---|---|---|
| **Features** (X) | Las variables de entrada | edad, horas_estudio, carrera, ... |
| **Target** (y) | Lo que queremos predecir | nota_final |
| **Observación** | Una fila | Un estudiante |

```
Modelo(features) → predicción de target
Modelo(edad, horas, carrera, ...) → nota_final predicha
```

> 💡 **No todas las columnas son features útiles.** El nombre del estudiante no ayuda a predecir la nota.

In [ ]:
# Selección de features: ¿qué columnas usamos?

print("📋 Todas las columnas del dataset:\n")
for col in df.columns:
    tipo = df[col].dtype
    unicos = df[col].nunique()
    util = "❌ No usar" if col in ["nombre"] else "✅ Feature" if col != "nota_final" else "🎯 TARGET"
    print(f"  {col:<20} tipo: {str(tipo):<10} únicos: {unicos:<6} → {util}")

print("\n📌 Separar features (X) del target (y):")
print("   X = todo menos 'nombre' y 'nota_final'")
print("   y = 'nota_final'")

In [ ]:
# Definir X (features) e y (target)

# Columnas que NO usamos como features
cols_excluir = ["nombre", "nota_final"]

# Features
feature_cols = [c for c in df.columns if c not in cols_excluir]
X = df[feature_cols].copy()
y = df["nota_final"].copy()

print(f"📊 Features (X): {X.shape}  → {list(X.columns)}")
print(f"🎯 Target  (y):  {y.shape}")
print(f"\nPrimeras 3 filas de X:")
X.head(3)

---
## 3. Features numéricas vs categóricas

| Tipo | Ejemplos | ¿El modelo puede usarla directamente? |
|---|---|---|
| **Numérica** | edad, horas_estudio | ✅ Sí |
| **Booleana** | trabaja | ✅ Sí (True=1, False=0) |
| **Categórica** | carrera | ❌ No → necesita encoding |

> 💡 **Un modelo de ML solo entiende números.** Las categorías tipo "Ingeniería" las convertimos con encoding.

In [ ]:
# Clasificar las features por tipo

print("📊 Clasificación de features:\n")

numericas = []
categoricas = []
booleanas = []

for col in X.columns:
    if X[col].dtype == "bool":
        booleanas.append(col)
    elif X[col].dtype in ["int64", "float64"]:
        numericas.append(col)
    else:
        categoricas.append(col)

print(f"  📐 Numéricas:   {numericas}")
print(f"  🔤 Categóricas: {categoricas}")
print(f"  ✓✗ Booleanas:   {booleanas}")

print(f"\n📌 Necesitamos transformar: {categoricas} (no son números)")

---
## 4. One-hot encoding

Para convertir una variable categórica en números, usamos **one-hot encoding**:
creamos una columna binaria (0/1) por cada categoría.

### Ejemplo visual

| carrera | → | carrera_Diseño | carrera_Economía | carrera_Ingeniería | carrera_Sistemas |
|---|---|---|---|---|---|
| Ingeniería | → | 0 | 0 | 1 | 0 |
| Diseño | → | 1 | 0 | 0 | 0 |
| Sistemas | → | 0 | 0 | 0 | 1 |

> 💡 **¿Por qué no usar números 1,2,3,4?** Porque el modelo interpretaría que Sistemas (4) > Ingeniería (1), lo cual no tiene sentido para categorías sin orden.

In [ ]:
# One-hot encoding paso a paso (para entender)

print("Antes del encoding:")
print(X["carrera"].value_counts())
print(f"\nTipo: {X['carrera'].dtype}")

In [ ]:
# Aplicar one-hot encoding con pandas

X_encoded = pd.get_dummies(X, columns=["carrera"], prefix="carrera", dtype=int)

print(f"📊 Antes del encoding: {X.shape[1]} columnas")
print(f"📊 Después:            {X_encoded.shape[1]} columnas")
print(f"\nColumnas nuevas: {[c for c in X_encoded.columns if 'carrera' in c]}")

# Convertir booleano a int
X_encoded["trabaja"] = X_encoded["trabaja"].astype(int)

print("\nPrimeras 5 filas del dataset codificado:")
X_encoded.head(5)

In [ ]:
# Verificar que TODAS las columnas son numéricas

print("📊 Tipos de datos después del encoding:\n")
for col in X_encoded.columns:
    tipo = X_encoded[col].dtype
    es_num = tipo in ["int64", "float64", "int32", "uint8"]
    emoji = "✅" if es_num else "❌"
    print(f"  {emoji} {col:<30} {tipo}")

todo_numerico = X_encoded.select_dtypes(include=["number"]).shape[1] == X_encoded.shape[1]
print(f"\n{'✅ Todo numérico — listo para ML!' if todo_numerico else '❌ Aún hay columnas no numéricas'}")

---
## 5. Normalización

Las features tienen **escalas muy diferentes**: edad va de 18 a 35, pero asistencia va de 0 a 100.
Muchos modelos funcionan mejor si las escalas son similares.

### Métodos comunes

| Método | Fórmula | Resultado | Cuándo usar |
|---|---|---|---|
| **Min-Max** | $(x - min) / (max - min)$ | Valores en [0, 1] | Redes neuronales |
| **Z-score** | $(x - \mu) / \sigma$ | Media=0, std=1 | Regresión, SVM |

> 💡 **No siempre hay que normalizar.** Árboles de decisión y random forest no lo necesitan.

In [ ]:
# Comparar las escalas ANTES de normalizar

cols_num = ["edad", "horas_estudio", "nota_parcial", "asistencia_pct"]

print("📊 Escalas originales:\n")
print(f"{'Columna':<20} {'Min':<10} {'Max':<10} {'Media':<10} {'Std'}")
print("─" * 60)
for col in cols_num:
    print(f"{col:<20} {X_encoded[col].min():<10.1f} {X_encoded[col].max():<10.1f} "
          f"{X_encoded[col].mean():<10.1f} {X_encoded[col].std():.1f}")

print("\n⚠️  Las escalas son MUY diferentes. Esto puede confundir a algunos modelos.")

In [ ]:
# Normalización Min-Max manual (para entender la fórmula)

X_norm = X_encoded.copy()

print("🔧 Aplicando normalización Min-Max a columnas numéricas:\n")

for col in cols_num:
    vmin = X_norm[col].min()
    vmax = X_norm[col].max()
    X_norm[col] = (X_norm[col] - vmin) / (vmax - vmin)
    print(f"  ✅ {col}: [{vmin:.1f}, {vmax:.1f}] → [0, 1]")

print("\n📊 Escalas después de normalizar:")
print(f"\n{'Columna':<20} {'Min':<10} {'Max':<10} {'Media':<10} {'Std'}")
print("─" * 60)
for col in cols_num:
    print(f"{col:<20} {X_norm[col].min():<10.2f} {X_norm[col].max():<10.2f} "
          f"{X_norm[col].mean():<10.2f} {X_norm[col].std():.2f}")

print("\n✅ Ahora todas las columnas están en la misma escala [0, 1].")

---
## 6. Train / Validation / Test

Antes de entrenar un modelo, hay que **dividir** los datos en 3 partes:

| Partición | % típico | ¿Para qué? |
|---|---|---|
| **Train** | 60-70% | El modelo aprende de estos datos |
| **Validation** | 15-20% | Para ajustar hiperparámetros durante el desarrollo |
| **Test** | 15-20% | Evaluación FINAL — se usa UNA sola vez |

### ¿Por qué dividir?

```
❌ Malo:  entrenar y evaluar con los MISMOS datos
          → el modelo "se aprende de memoria" las respuestas
          → funciona perfecto en train pero falla con datos nuevos

✅ Bueno: entrenar con unos datos, evaluar con OTROS
          → si funciona bien en datos que nunca vio = generaliza
```

> 💡 **Analogía del examen:** sería como dar el examen con las mismas preguntas que practicaste. No demuestra que aprendiste.

In [ ]:
# División manual (para entender el concepto)

# Mezclar los datos (importante para que no haya sesgo de orden)
indices = list(range(len(X_norm)))
random.seed(42)
random.shuffle(indices)

# Calcular los cortes
n = len(indices)
corte_train = int(n * 0.70)  # 70% train
corte_val   = int(n * 0.85)  # 15% validation
# El resto (15%) es test

# Dividir
idx_train = indices[:corte_train]
idx_val   = indices[corte_train:corte_val]
idx_test  = indices[corte_val:]

X_train = X_norm.iloc[idx_train].reset_index(drop=True)
X_val   = X_norm.iloc[idx_val].reset_index(drop=True)
X_test  = X_norm.iloc[idx_test].reset_index(drop=True)

y_train = y.iloc[idx_train].reset_index(drop=True)
y_val   = y.iloc[idx_val].reset_index(drop=True)
y_test  = y.iloc[idx_test].reset_index(drop=True)

print("📊 Partición de datos:")
print(f"\n  {'Partición':<15} {'Filas':<10} {'%'}")
print("  " + "─" * 35)
print(f"  {'Train':<15} {len(X_train):<10} {len(X_train)/n*100:.0f}%")
print(f"  {'Validation':<15} {len(X_val):<10} {len(X_val)/n*100:.0f}%")
print(f"  {'Test':<15} {len(X_test):<10} {len(X_test)/n*100:.0f}%")
print(f"  {'TOTAL':<15} {n:<10} 100%")

print("\n📊 Distribución del target por partición:")
print(f"  Train      — media: {y_train.mean():.2f}, std: {y_train.std():.2f}")
print(f"  Validation — media: {y_val.mean():.2f}, std: {y_val.std():.2f}")
print(f"  Test       — media: {y_test.mean():.2f}, std: {y_test.std():.2f}")
print("\n💡 Las medias son similares → la partición es representativa. ✅")

In [ ]:
# Visualizar la partición

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, data, nombre, color in [
    (axes[0], y_train, f"Train ({len(y_train)})", "#3498db"),
    (axes[1], y_val,   f"Validation ({len(y_val)})", "#f39c12"),
    (axes[2], y_test,  f"Test ({len(y_test)})", "#e74c3c"),
]:
    ax.hist(data, bins=15, color=color, edgecolor="#333", alpha=0.8)
    ax.axvline(data.mean(), color="red", linestyle="--", linewidth=1.5)
    ax.set_title(nombre, fontsize=12, fontweight="bold")
    ax.set_xlabel("nota_final")
    ax.grid(True, alpha=0.2)

fig.suptitle("Distribución de nota_final en cada partición",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("📊 Las 3 distribuciones son similares → partición bien hecha.")

---
## 7. Primera predicción con regresión lineal

Ahora que tenemos features preparadas y datos particionados, ¡hagamos nuestra primera predicción real!

Usamos **regresión lineal**: el modelo más simple de ML.

```
nota_final = w₁×edad + w₂×horas + w₃×nota_parcial + ... + b
```

In [ ]:
# Regresión lineal desde cero (sin sklearn, para entender)

# Convertir a listas para cálculos simples
# Usamos solo 3 features principales para simplicidad
features_modelo = ["horas_estudio", "nota_parcial", "asistencia_pct"]

# Mini implementación de regresión lineal con gradiente descendente
def entrenar_regresion(X_df, y_series, features, lr=0.01, epochs=500):
    """Regresión lineal con gradiente descendente."""
    X_vals = X_df[features].values.tolist()
    y_vals = y_series.values.tolist()
    n = len(y_vals)
    n_features = len(features)
    
    # Inicializar pesos en 0
    w = [0.0] * n_features
    b = 0.0
    
    historial_loss = []
    
    for epoch in range(epochs):
        # Forward: calcular predicciones
        predicciones = []
        for i in range(n):
            pred = b
            for j in range(n_features):
                pred += w[j] * X_vals[i][j]
            predicciones.append(pred)
        
        # Loss: error cuadrático medio (MSE)
        mse = sum((predicciones[i] - y_vals[i])**2 for i in range(n)) / n
        historial_loss.append(mse)
        
        # Gradientes
        grad_w = [0.0] * n_features
        grad_b = 0.0
        for i in range(n):
            error = predicciones[i] - y_vals[i]
            for j in range(n_features):
                grad_w[j] += (2.0 / n) * error * X_vals[i][j]
            grad_b += (2.0 / n) * error
        
        # Actualizar pesos
        for j in range(n_features):
            w[j] -= lr * grad_w[j]
        b -= lr * grad_b
    
    return w, b, historial_loss

# Entrenar
print("🚀 Entrenando regresión lineal desde cero...\n")
w, b, historial = entrenar_regresion(X_train, y_train, features_modelo, lr=0.1, epochs=300)

print("📊 Pesos aprendidos:")
for feat, peso in zip(features_modelo, w):
    print(f"  {feat:<20} w = {peso:+.4f}")
print(f"  {'bias':<20} b = {b:+.4f}")
print(f"\n📊 Loss final (MSE): {historial[-1]:.4f}")

In [ ]:
# Curva de entrenamiento

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(historial, color="#e74c3c", linewidth=2)
ax.set_xlabel("Época", fontsize=12)
ax.set_ylabel("MSE (Error cuadrático medio)", fontsize=12)
ax.set_title("Curva de entrenamiento: el error baja con cada época", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3)

# Marcar inicio y final
ax.scatter([0], [historial[0]], color="red", s=100, zorder=5, label=f"Inicio: MSE={historial[0]:.2f}")
ax.scatter([len(historial)-1], [historial[-1]], color="green", s=100, zorder=5, label=f"Final: MSE={historial[-1]:.4f}")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("📊 La curva baja → el modelo está aprendiendo.")
print("   Cuando se aplana → ya aprendió lo que puede.")

In [ ]:
# Evaluar en validation y test

def predecir(X_df, features, w, b):
    """Hacer predicciones con los pesos aprendidos."""
    X_vals = X_df[features].values.tolist()
    preds = []
    for row in X_vals:
        pred = b + sum(w[j] * row[j] for j in range(len(w)))
        preds.append(pred)
    return preds

def calcular_mae(reales, predichos):
    """Error absoluto medio."""
    return sum(abs(r - p) for r, p in zip(reales, predichos)) / len(reales)

# Predicciones en cada partición
pred_train = predecir(X_train, features_modelo, w, b)
pred_val   = predecir(X_val,   features_modelo, w, b)
pred_test  = predecir(X_test,  features_modelo, w, b)

mae_train = calcular_mae(y_train.tolist(), pred_train)
mae_val   = calcular_mae(y_val.tolist(),   pred_val)
mae_test  = calcular_mae(y_test.tolist(),  pred_test)

print("📊 Evaluación del modelo (MAE = error promedio en puntos):\n")
print(f"  {'Partición':<15} {'MAE':<10} {'Interpretación'}")
print("  " + "─" * 55)
print(f"  {'Train':<15} {mae_train:<10.3f} Error promedio en datos de entrenamiento")
print(f"  {'Validation':<15} {mae_val:<10.3f} Error en datos NO vistos")
print(f"  {'Test':<15} {mae_test:<10.3f} Error en evaluación FINAL")

# Diagnóstico
diff = abs(mae_train - mae_val)
if diff < 0.3:
    print("\n✅ Train y Validation similares → el modelo generaliza bien.")
elif mae_train < mae_val:
    print("\n⚠️  Validation peor que Train → posible OVERFITTING.")
else:
    print("\n🤔 Train peor que Validation → datos de train más difíciles.")

---
## 8. Ejercicio evaluable

### Consigna

Aplicá encoding, normalización y partición a un nuevo conjunto de features.

### Criterio de aprobación

- ✅ One-hot encoding aplicado correctamente
- ✅ Normalización Min-Max aplicada
- ✅ Partición 70/15/15 implementada
- ✅ Evaluación en 3 particiones con MAE
- ✅ Reflexión sobre los resultados

In [ ]:
# ══════════════════════════════════════════════════════════════
#  EJERCICIO EVALUABLE — Features, encoding y partición
# ══════════════════════════════════════════════════════════════

# ── Parte 1: preguntas conceptuales ──────────────────────────

respuestas = {
    # ¿Por qué no podemos meter "carrera" directamente en el modelo?
    "q1_por_que_encoding": "",  # TODO
    
    # ¿Qué pasaría si codificamos carrera como 1,2,3,4?
    "q2_numeros_vs_onehot": "",  # TODO
    
    # ¿Por qué separamos en train/val/test?
    "q3_por_que_split": "",  # TODO
    
    # ¿Qué pasa si entrenamos y evaluamos con los mismos datos?
    "q4_mismo_dato": "",  # TODO
}

vacias = [k for k, v in respuestas.items() if v.strip() == ""]
if vacias:
    print(f"⚠️  Faltan {len(vacias)} respuestas: {vacias}")
else:
    print("✅ Todas las respuestas completadas.")

In [ ]:
# ── Parte 2: tu pipeline completo ───────────────────────────
#
# Partí del dataset original (df) y hacé:
# 1. Seleccionar features (excluyendo 'nombre' y 'nota_final')
# 2. Aplicar one-hot encoding a 'carrera'
# 3. Convertir 'trabaja' a int
# 4. Normalizar las columnas numéricas con Min-Max
# 5. Dividir en train (70%) / val (15%) / test (15%)
#
# TODO: escribí tu código abajo

# Paso 1:

# Paso 2:

# Paso 3:

# Paso 4:

# Paso 5:

# Verificación (descomenta cuando termines):
# print(f"X_train: {X_train.shape}")
# print(f"X_val:   {X_val.shape}")
# print(f"X_test:  {X_test.shape}")
# print(f"Todo numérico: {X_train.select_dtypes(include=['number']).shape[1] == X_train.shape[1]}")

print("\n🎉 Guardá este notebook y envialo como entrega de la Clase 5.")

---
## Resumen de la clase

| Concepto | Lo que aprendiste |
|---|---|
| Features (X) | Variables de entrada para el modelo |
| Target (y) | Lo que queremos predecir |
| One-hot encoding | Convertir categorías en columnas binarias 0/1 |
| Normalización | Llevar todas las features a la misma escala [0,1] |
| Train/Val/Test | 70/15/15 — entrenar, ajustar, evaluar |
| Regresión lineal | Modelo que aprende pesos con gradiente descendente |

### Próxima clase

En la **Clase 6** vamos a profundizar en **métricas de evaluación**: MAE, RMSE, R², sesgo vs varianza.

> 📌 **Recordá:** un modelo solo puede usar features numéricas. Encoding + normalización son pasos obligatorios.